In [ ]:
import pandas as pd
import numpy as np

INPUT  = "SPOC_analysis_results.csv"
OUTPUT = "SPOC_ls_results_with_flags.csv"

#Thresholds
POWER_THRESHOLD = 0.05
PERIOD_TOL_FRAC = 0.10

DD_PERIOD_TOL  = 0.15
DD_POWER_RATIO = 0.5

#column names
TIC  = "ticid"

P1   = "peak1_period"
POW1 = "peak1_power"

P2   = "peak2_period"
POW2 = "peak2_power"


df = pd.read_csv(INPUT)

print("Loaded rows:", len(df))


# Power threshold flag
df["power_pass"] = df[POW1] > POWER_THRESHOLD


# Period agreement across sectors

def compute_period_agreement(group):

    idx_ref = group[POW1].idxmax()
    ref_period = group.loc[idx_ref, P1]

    frac_diff = np.abs(group[P1] - ref_period) / ref_period

    group["ref_period"] = ref_period
    group["period_agree"] = frac_diff <= PERIOD_TOL_FRAC

    return group


df = df.groupby(TIC, group_keys=False).apply(compute_period_agreement)


#Combined criteria

df["power_and_agree"] = df["power_pass"] & df["period_agree"]


#Douple dipper check

def check_double_dipper_row(row):

    p1  = row[P1]
    p2  = row[P2]
    pw1 = row[POW1]
    pw2 = row[POW2]

    if np.isnan(p1) or np.isnan(p2):
        return pd.Series([False, p1])

    period_match = abs(p2 / p1 - 2.0) < DD_PERIOD_TOL * 2.0
    power_ok     = pw2 >= DD_POWER_RATIO * pw1

    if period_match and power_ok:
        return pd.Series([True, p2])

    return pd.Series([False, p1])


df[["double_dipper", "true_period_d"]] = df.apply(
    check_double_dipper_row,
    axis=1
)


# Save
df.to_csv(OUTPUT, index=False)

print("Saved:", OUTPUT)

print("\nSummary")
print("power_pass:", df["power_pass"].sum())
print("period_agree:", df["period_agree"].sum())
print("power_and_agree:", df["power_and_agree"].sum())
print("double_dipper:", df["double_dipper"].sum())